In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# TODO: TwoHot over rewards with reward and continue models
# TODO: Implement losses
# TODO: Allow vectorized buffer storage and sampling

In [3]:
import gymnasium as gym
import numpy as np
import torch

import fishyrl as frl

In [4]:
# Create environment
num_envs = 4
envs = gym.vector.AsyncVectorEnv([
    lambda: gym.make('CartPole-v1'),
    lambda: gym.make('CartPole-v1'),
    lambda: gym.make('CartPole-v1'),
    lambda: gym.make('CartPole-v1'),
])

In [5]:
# Initialize model
OBS_DIM = 4
MODEL_ACTIONS = [frl.actions.DiscreteAction(2)]
ACTION_DIM = sum([action.output_dim for action in MODEL_ACTIONS])

EMBEDDED_OBS_DIM = 4

STOCHASTIC_DIM = 6
DETERMINISTIC_DIM = 7
CATEGORICAL_BINS = 11
REWARD_BINS = 13

# Seed
torch.random.manual_seed(42)

# Encoder-decoder models
encoder_model = frl.models.MLPEncoder(OBS_DIM, EMBEDDED_OBS_DIM)
decoder_model = frl.models.MLPDecoder(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, OBS_DIM)

# RSSM models
recurrent_model = frl.models.RecurrentModel(STOCHASTIC_DIM * CATEGORICAL_BINS + ACTION_DIM, DETERMINISTIC_DIM)
representation_model = frl.models.MLP(EMBEDDED_OBS_DIM + DETERMINISTIC_DIM, STOCHASTIC_DIM * CATEGORICAL_BINS)
transition_model = frl.models.MLP(DETERMINISTIC_DIM, STOCHASTIC_DIM * CATEGORICAL_BINS)
rssm_model = frl.models.RSSM(recurrent_model, representation_model, transition_model, CATEGORICAL_BINS)

# Reward and continue models
reward_model = frl.models.MLP(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, REWARD_BINS, [512])  # Double-check num layers
continue_model = frl.models.MLP(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, 1, [512])  # Double-check num layers

# Actor and critic models
actor_model = frl.models.Actor(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, MODEL_ACTIONS)
critic_model = frl.models.MLP(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, REWARD_BINS, [512])

# Buffer
buffer = frl.buffers.IndependentVectorizedBuffer(num_envs, 20)  # Buffer size

In [ ]:
start_train_step = 10
gradient_steps = 2
batch_size = 64
sequence_length = 5

def train(batch: dict[str, torch.Tensor]) -> None:
    """Single-step training function."""
    # Infer batch size and sequence length (batch, sequence_length, ...)
    batch_size, sequence_length = batch['obs'].shape[:2]

    # Embed observations
    embedded_obs = encoder_model(batch['obs'])

    # Initialize storage
    hidden_states = torch.empty((batch_size, sequence_length, DETERMINISTIC_DIM))
    priors = torch.empty((batch_size, sequence_length, STOCHASTIC_DIM * CATEGORICAL_BINS))
    priors_logits = torch.empty((batch_size, sequence_length, STOCHASTIC_DIM * CATEGORICAL_BINS))
    posteriors = torch.empty((batch_size, sequence_length, STOCHASTIC_DIM * CATEGORICAL_BINS))
    posteriors_logits = torch.empty((batch_size, sequence_length, STOCHASTIC_DIM * CATEGORICAL_BINS))

    # Compute model outputs for each time step, starting with initial recurrent state and posteriors
    for i in range(sequence_length):
        # Run through recurrent model
        ret = rssm_model(
            batch['actions'][:, i-1] if i > 0 else None,  # Use the action from the previous step, like in the environment loop
            posteriors[:, i-1] if i > 0 else None,
            hidden_states[:, i-1] if i > 0 else None,
            embedded_obs[:, i],
            batch['terminations'][:, i-1] | batch['truncations'][:, i-1] if i > 0 else None,  # Get initializations using result of previous step
            batch_dim=batch_size,
        )
        hidden_states[:, i] = ret['hidden_state']
        priors[:, i] = ret['prior']
        priors_logits[:, i] = ret['prior_logits']
        posteriors[:, i] = ret['posterior']
        posteriors_logits[:, i] = ret['posterior_logits']

def main() -> None:
    """Main training loop."""
    # Initialize variables
    actions = posteriors = hidden_states = initializations = None
    rewards, terminations, truncations = np.zeros(num_envs), np.zeros(num_envs, dtype=bool), np.zeros(num_envs, dtype=bool)
    pred_rewards = pred_continues = 0

    # Loop for specified number of iterations
    obs, info = envs.reset(seed=42)
    for environment_step in range(100):
        ## Compute an environment step
        # Embed observation
        embedded_obs = encoder_model(torch.from_numpy(obs))  # TODO: GPU

        # Compute hidden states
        ret = rssm_model(
            actions,
            posteriors,
            hidden_states,
            embedded_obs,
            initializations,
            batch_dim=envs.num_envs,
        )
        hidden_states = ret['hidden_state']
        priors = ret['prior']
        prior_logits = ret['prior_logits']
        posteriors = ret['posterior']
        posterior_logits = ret['posterior_logits']

        # Decode observation
        pred_obs = decoder_model(torch.cat((posteriors, hidden_states), dim=-1))

        # Predict reward and continue
        # TODO: Double-check, but compute reward for the previous action
        pred_rewards = reward_model(torch.cat((posteriors, hidden_states), dim=-1))
        pred_continues = continue_model(torch.cat((posteriors, hidden_states), dim=-1))

        # Compute actions
        actions, action_distributions = actor_model(torch.cat((posteriors, hidden_states), dim=-1))
        frl.actions.simplify_actions(actions, MODEL_ACTIONS).flatten()

        # Record to buffer
        buffer.add({
            # Environment-related experiences
            'obs': obs,
            'rewards': rewards,
            'terminations': terminations,
            'truncations': truncations,
            # Predicted environment experiences
            'pred_rewards': pred_rewards.detach().cpu().numpy(),
            'pred_continues': pred_continues.detach().cpu().numpy(),
            'pred_obs': pred_obs.detach().cpu().numpy(),
            # RSSM experience
            'hidden_states': hidden_states.detach().cpu().numpy(),
            'priors': priors.detach().cpu().numpy(),
            'prior_logits': prior_logits.detach().cpu().numpy(),
            'posteriors': posteriors.detach().cpu().numpy(),
            'posterior_logits': posterior_logits.detach().cpu().numpy(),
            # Actions
            'actions': actions.detach().cpu().numpy(),
            # 'action_distributions': action_distributions,
        })

        # Step environment
        obs, rewards, terminations, truncations, infos = envs.step(
            frl.actions.simplify_actions(actions, MODEL_ACTIONS).flatten().numpy())  # Flatten will be removed here when using more complex action spaces
        initializations = torch.tensor(terminations | truncations, dtype=torch.bool, device='cpu')  # TODO: Detect device

        ## Train
        if environment_step >= start_train_step:
            # TODO: Implement or evaluate ratio
            for gradient_step in range(gradient_steps):
                # Sample batch of experiences from buffer and train
                batch = buffer.sample(batch_size, sequence_length=sequence_length)
                batch = frl.buffers.convert_samples_to_tensors(batch, device='cpu')  # TODO: Use GPU when available
                train(batch)

In [7]:
main()

In [8]:
{k: v.shape for k, v in buffer.sample(64, sequence_length=5).items()}

{'obs': (64, 5, 4),
 'rewards': (64, 5),
 'terminations': (64, 5),
 'truncations': (64, 5),
 'pred_rewards': (64, 5, 13),
 'pred_continues': (64, 5, 1),
 'pred_obs': (64, 5, 4),
 'hidden_states': (64, 5, 7),
 'priors': (64, 5, 66),
 'prior_logits': (64, 5, 66),
 'posteriors': (64, 5, 66),
 'posterior_logits': (64, 5, 66),
 'actions': (64, 5, 2)}